# 메이플스토리 4종 문서 통합 ChromaDB

공식 가이드·직업·아이템·인벤 팁 청크를 공유 metadata 스키마로 정규화하고, 기존 `maplestory_guides` 컬렉션을 보존한 채 `maplestory_knowledge` 컬렉션에 통합합니다. 마지막 셀의 `retrieve(question, k=5)`는 팀 RAG Chain에서 그대로 사용할 수 있습니다.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import gc
import hashlib
import json
import os
import tempfile

import chromadb
import numpy as np
import pandas as pd
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from IPython.display import display
from langchain_core.documents import Document

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'data/RAG').is_dir() and (candidate / 'ㅋㅌㅊ').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
RAG_DATA_DIR = PROJECT_ROOT / 'data/RAG'
INVEN_RAG_DIR = PROJECT_ROOT / 'ㅋㅌㅊ/output/RAG'
CHROMA_DIR = PROJECT_ROOT / 'chroma_db'
COLLECTION_NAME = 'maplestory_knowledge'
LEGACY_COLLECTION_NAME = 'maplestory_guides'
MODEL_NAME = 'jhgan/ko-sroberta-multitask'
BATCH_SIZE = 256
REPORT_PATH = RAG_DATA_DIR / 'maplestory_integrated_chromadb_report.json'
REQUIRED_METADATA = (
    'source', 'name', 'section_title', 'article_id',
    'board_id', 'url', 'chunk_index', 'total_chunks',
)

print('프로젝트 루트:', PROJECT_ROOT)
print('통합 DB:', CHROMA_DIR)
print('통합 컬렉션:', COLLECTION_NAME)

프로젝트 루트: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3
통합 DB: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\chroma_db
통합 컬렉션: maplestory_knowledge


## 공유 metadata 스키마

모든 Chroma 문서는 `source`, `name`, `section_title`, `article_id`, `board_id`, `url`, `chunk_index`, `total_chunks`를 반드시 가집니다. `article_id`와 `board_id`는 원천별 타입 차이로 필터가 흔들리지 않도록 문자열로 통일합니다.

In [2]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def stable_hash(text):
    return hashlib.sha1(str(text).encode('utf-8')).hexdigest()[:16]

def scalar_metadata(metadata):
    return {
        str(key): value for key, value in metadata.items()
        if value is not None and isinstance(value, (str, int, float, bool))
    }

def validate_cosine_collection(chroma_collection):
    metric = (chroma_collection.metadata or {}).get('hnsw:space')
    if metric != 'cosine':
        raise ValueError(f"컬렉션 {chroma_collection.name}의 거리 함수가 cosine이 아닙니다: {metric}")
    return True

def source_identity(raw_metadata, source):
    if source == 'guide':
        return {'group_key': str(raw_metadata['article_id']), 'article_id': str(raw_metadata['article_id']), 'board_id': str(raw_metadata['board_id']), 'name': str(raw_metadata['name']).strip(), 'section_title': str(raw_metadata['section_title']).strip(), 'url': str(raw_metadata['url']).strip()}
    if source == 'jobs':
        job_id = str(raw_metadata['job_id'])
        return {'group_key': job_id, 'article_id': job_id, 'board_id': 'official_jobs', 'name': str(raw_metadata['name']).strip(), 'section_title': str(raw_metadata.get('category') or '직업').strip(), 'url': str(raw_metadata['url']).strip()}
    if source == 'items':
        url = str(raw_metadata['url']).strip()
        return {'group_key': url, 'article_id': f'item_{stable_hash(url)}', 'board_id': 'official_items', 'name': str(raw_metadata.get('name') or '아이템 정보').strip(), 'section_title': str(raw_metadata.get('section_title') or raw_metadata.get('name') or '아이템').strip(), 'url': url}
    if source == 'inven_tip':
        article_id = str(raw_metadata['article_id'])
        return {'group_key': str(raw_metadata.get('document_id') or article_id), 'article_id': article_id, 'board_id': '2304', 'name': str(raw_metadata['name']).strip(), 'section_title': str(raw_metadata.get('section_title') or '기타').strip(), 'url': str(raw_metadata['url']).strip()}
    raise ValueError(f'지원하지 않는 source: {source}')

def normalize_source_documents(raw_documents, source):
    identities = [source_identity(doc['metadata'], source) for doc in raw_documents]
    totals = Counter(identity['group_key'] for identity in identities)
    seen = Counter()
    normalized = []
    for raw_document, identity in zip(raw_documents, identities):
        group_key = identity['group_key']
        chunk_index = seen[group_key]
        seen[group_key] += 1
        raw_metadata = raw_document['metadata']
        metadata = scalar_metadata(raw_metadata)
        if raw_metadata.get('source') and raw_metadata.get('source') != source:
            metadata['origin_source'] = str(raw_metadata['source'])
        metadata.update({'source': source, 'name': identity['name'], 'section_title': identity['section_title'], 'article_id': identity['article_id'], 'board_id': identity['board_id'], 'url': identity['url'], 'chunk_index': chunk_index, 'total_chunks': totals[group_key]})
        integrated_id = f"{source}:{identity['article_id']}:{chunk_index}"
        normalized.append({'id': integrated_id, 'page_content': str(raw_document['page_content']).strip(), 'metadata': metadata})
    return normalized

def validate_shared_schema(documents):
    errors = []
    ids = [doc['id'] for doc in documents]
    if len(ids) != len(set(ids)):
        errors.append(f'중복 통합 ID: {len(ids) - len(set(ids))}건')
    grouped = defaultdict(list)
    for document in documents:
        metadata = document['metadata']
        missing = [key for key in REQUIRED_METADATA if metadata.get(key) in (None, '')]
        if missing:
            errors.append(f"{document['id']} 필드 누락: {missing}")
        if not document['page_content']:
            errors.append(f"{document['id']} 본문 없음")
        invalid_types = [key for key, value in metadata.items() if not isinstance(value, (str, int, float, bool))]
        if invalid_types:
            errors.append(f"{document['id']} 비스칼라 metadata: {invalid_types}")
        grouped[(metadata.get('source'), metadata.get('article_id'))].append(metadata)
    for group_key, rows in grouped.items():
        expected_total = len(rows)
        totals = {row.get('total_chunks') for row in rows}
        indices = sorted(row.get('chunk_index') for row in rows)
        if totals != {expected_total}:
            errors.append(f'{group_key} total_chunks 불일치: {totals}, 실제 {expected_total}')
        if indices != list(range(expected_total)):
            errors.append(f'{group_key} chunk_index 불연속')
    if errors:
        raise ValueError('공유 스키마 검증 실패\n' + '\n'.join(errors[:20]))
    return dict(sorted(Counter(doc['metadata']['source'] for doc in documents).items()))

In [3]:
guide_raw = read_json(RAG_DATA_DIR / 'maple_guides_documents_chunked.json')
jobs_raw = read_json(RAG_DATA_DIR / 'maple_jobs_documents.json')
items_raw = read_json(RAG_DATA_DIR / 'maple_items_documents.json')
inven_chunks_path = INVEN_RAG_DIR / 'maple_inven_tips_documents_chunked.json'
inven_embeddings_path = INVEN_RAG_DIR / 'maple_inven_tips_embeddings.npy'
inven_manifest_path = INVEN_RAG_DIR / 'maple_inven_tips_embeddings_manifest.json'
inven_raw = read_json(inven_chunks_path)
inven_manifest = read_json(inven_manifest_path)
inven_vectors = np.load(inven_embeddings_path, allow_pickle=False)
if inven_manifest.get('model_name') != MODEL_NAME:
    raise ValueError(f"인벤 임베딩 모델 불일치: {inven_manifest.get('model_name')} != {MODEL_NAME}")
if inven_manifest.get('normalized') is not True:
    raise ValueError('인벤 임베딩 manifest의 normalized가 True가 아닙니다.')
if inven_manifest.get('embedding_dimension') != 768:
    raise ValueError(f"인벤 임베딩 차원 불일치: {inven_manifest.get('embedding_dimension')}")

guide_documents = normalize_source_documents(guide_raw, 'guide')
jobs_documents = normalize_source_documents(jobs_raw, 'jobs')
items_documents = normalize_source_documents(items_raw, 'items')
inven_documents = normalize_source_documents(inven_raw, 'inven_tip')
official_documents = guide_documents + jobs_documents + items_documents
all_documents = official_documents + inven_documents
source_counts = validate_shared_schema(all_documents)

assert [doc['id'] for doc in inven_raw] == inven_manifest['chunk_ids']
assert sha256_file(inven_chunks_path) == inven_manifest['chunks_sha256']
assert sha256_file(inven_embeddings_path) == inven_manifest['embeddings_sha256']
assert inven_vectors.shape == (len(inven_documents), inven_manifest['embedding_dimension'])
assert inven_vectors.dtype == np.float32 and np.isfinite(inven_vectors).all()
assert np.allclose(np.linalg.norm(inven_vectors, axis=1), 1.0, atol=1e-5)

display({'공유 스키마 검증': '통과', '통합 청크': len(all_documents), 'source별 청크': source_counts, '고유 ID': len({doc['id'] for doc in all_documents})})
display(pd.DataFrame([{key: doc['metadata'][key] for key in REQUIRED_METADATA} for doc in [guide_documents[0], jobs_documents[0], items_documents[0], inven_documents[0]]]))

{'공유 스키마 검증': '통과',
 '통합 청크': 9200,
 'source별 청크': {'guide': 1441, 'inven_tip': 5506, 'items': 2205, 'jobs': 48},
 '고유 ID': 9200}

,source,name,section_title,article_id,board_id,url,chunk_index,total_chunks
0,guide,게임 시작,기초 가이드,272,429467337,https://maplestory.nexon.com/Guide/N23GameInfo...,0,6
1,jobs,히어로,전사,1,official_jobs,https://maplestory.nexon.com/Guide/N23Job/View/1,0,1
2,items,로얄스타일,로얄스타일,item_545b53d9f225a2aa,official_items,https://maplestory.nexon.com/Guide/CashShop/Pr...,0,28
3,inven_tip,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,실험,48082,2304,https://www.inven.co.kr/board/maple/2304/48082,0,42


## 기존 컬렉션 보존 + 통합 컬렉션 적재

기존 `maplestory_guides`의 공식 3종 임베딩이 온전하면 재사용하고, 없거나 문서 순서가 다르면 같은 모델로 다시 계산합니다. `maplestory_knowledge`는 stable ID로 upsert하므로 반복 실행해도 중복 적재되지 않습니다.

In [4]:
previous_client = globals().pop('client', None)
globals().pop('collection', None)
if previous_client is not None:
    previous_client.close()
    del previous_client
    gc.collect()

embedding_function = SentenceTransformerEmbeddingFunction(model_name=MODEL_NAME, normalize_embeddings=True)
client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection_names = {item.name for item in client.list_collections()}

def load_legacy_official_vectors():
    if LEGACY_COLLECTION_NAME not in collection_names:
        return None, '기존 컬렉션 없음'
    legacy = client.get_collection(LEGACY_COLLECTION_NAME)
    if legacy.count() != len(official_documents):
        return None, f'기존 컬렉션 수 불일치({legacy.count()})'
    vector_by_id, document_by_id = {}, {}
    legacy_ids = [f'chunk_{index}' for index in range(len(official_documents))]
    for start in range(0, len(legacy_ids), BATCH_SIZE):
        result = legacy.get(ids=legacy_ids[start:start + BATCH_SIZE], include=['embeddings', 'documents'])
        if result.get('embeddings') is None:
            return None, '기존 임베딩 조회 실패'
        for legacy_id, vector, text in zip(result['ids'], result['embeddings'], result['documents']):
            vector_by_id[legacy_id] = vector
            document_by_id[legacy_id] = text
    if len(vector_by_id) != len(official_documents):
        return None, '기존 ID 누락'
    for index, document in enumerate(official_documents):
        if document_by_id.get(f'chunk_{index}') != document['page_content']:
            return None, f'기존 문서 순서 불일치({index})'
    vectors = np.asarray([vector_by_id[f'chunk_{index}'] for index in range(len(official_documents))], dtype=np.float32)
    if vectors.shape != (len(official_documents), inven_manifest['embedding_dimension']):
        return None, f'기존 벡터 shape 불일치{vectors.shape}'
    if not np.isfinite(vectors).all():
        return None, '기존 벡터에 NaN/Inf 존재'
    if not np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5):
        return None, '기존 벡터 L2 정규화 불일치'
    sample_indices = np.linspace(0, len(official_documents) - 1, num=min(8, len(official_documents)), dtype=int)
    sample_texts = [official_documents[index]['page_content'] for index in sample_indices]
    reference_vectors = np.asarray(embedding_function(sample_texts), dtype=np.float32)
    sample_similarities = np.sum(reference_vectors * vectors[sample_indices], axis=1)
    if not np.all(sample_similarities >= 0.9999):
        return None, f'기존 벡터 모델 표본 불일치(min cosine={sample_similarities.min():.6f})'
    return vectors, '기존 maplestory_guides 벡터 재사용'

official_vectors, official_vector_source = load_legacy_official_vectors()
if official_vectors is None:
    encoded = []
    for start in range(0, len(official_documents), BATCH_SIZE):
        encoded.extend(embedding_function([doc['page_content'] for doc in official_documents[start:start + BATCH_SIZE]]))
    official_vectors = np.asarray(encoded, dtype=np.float32)
    official_vector_source = '동일 모델로 공식 3종 재임베딩'

if COLLECTION_NAME in collection_names:
    collection = client.get_collection(COLLECTION_NAME, embedding_function=embedding_function)
    validate_cosine_collection(collection)
else:
    collection = client.create_collection(COLLECTION_NAME, embedding_function=embedding_function, metadata={'hnsw:space': 'cosine', 'description': '메이플 공식 3종 + 인벤 팁 통합'})
    validate_cosine_collection(collection)

def upsert_documents(documents, vectors):
    for start in range(0, len(documents), BATCH_SIZE):
        batch = documents[start:start + BATCH_SIZE]
        end = start + len(batch)
        collection.upsert(ids=[doc['id'] for doc in batch], documents=[doc['page_content'] for doc in batch], metadatas=[doc['metadata'] for doc in batch], embeddings=np.asarray(vectors[start:end], dtype=np.float32).tolist())

upsert_documents(official_documents, official_vectors)
upsert_documents(inven_documents, inven_vectors)
target_ids = {doc['id'] for doc in all_documents}
stored_ids = set(collection.get(include=[])['ids'])
stale_ids = sorted(stored_ids - target_ids)
for start in range(0, len(stale_ids), BATCH_SIZE):
    collection.delete(ids=stale_ids[start:start + BATCH_SIZE])
assert collection.count() == len(all_documents)
assert set(collection.get(include=[])['ids']) == target_ids

display({'기존 컬렉션 보존': LEGACY_COLLECTION_NAME in {item.name for item in client.list_collections()}, '통합 컬렉션': COLLECTION_NAME, '적재 청크': collection.count(), '공식 벡터': official_vector_source, '인벤 벡터': '04_embedding.ipynb NPY 재사용', '삭제한 stale ID': len(stale_ids)})

C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4985.08it/s]

{'기존 컬렉션 보존': True,
 '통합 컬렉션': 'maplestory_knowledge',
 '적재 청크': 9200,
 '공식 벡터': '기존 maplestory_guides 벡터 재사용',
 '인벤 벡터': '04_embedding.ipynb NPY 재사용',
 '삭제한 stale ID': 0}

## 기존 RAG Chain 연결용 Retriever

`retrieve(question, k=5)`는 `List[Document]`를 반환합니다. 팀의 `build_context`와 전체 RAG Chain에서 별도 변환 없이 호출할 수 있습니다.

In [5]:
def retrieve(question, k=5, source=None):
    if not str(question).strip():
        raise ValueError('질문을 입력하세요.')
    if isinstance(k, bool) or not isinstance(k, int) or k < 1:
        raise ValueError('k는 1 이상의 정수여야 합니다.')
    collection_count = collection.count()
    candidate_k = min(max(k * 8, 40), collection_count)
    query_kwargs = {'query_texts': [str(question).strip()], 'include': ['documents', 'metadatas', 'distances']}
    if source is not None:
        query_kwargs['where'] = {'source': source}
    while True:
        query_kwargs['n_results'] = candidate_k
        response = collection.query(**query_kwargs)
        returned_ids = response['ids'][0]
        unique_articles = {(meta['source'], str(meta['article_id'])) for meta in response['metadatas'][0]}
        if len(unique_articles) >= k or len(returned_ids) < candidate_k or candidate_k >= collection_count:
            break
        candidate_k = min(candidate_k * 2, collection_count)
    selected, seen_articles = [], set()
    for text, metadata, distance in zip(response['documents'][0], response['metadatas'][0], response['distances'][0]):
        article_key = (metadata['source'], str(metadata['article_id']))
        if article_key in seen_articles:
            continue
        seen_articles.add(article_key)
        result_metadata = dict(metadata)
        result_metadata['distance'] = float(distance)
        result_metadata['similarity'] = 1.0 - float(distance)
        selected.append(Document(page_content=text, metadata=result_metadata))
        if len(selected) >= k:
            break
    return selected

SMOKE_QUERY = '스타포스 파괴 후 확정 복구는 어떻게 사용하나요?'
smoke_results = retrieve(SMOKE_QUERY, k=5)
assert len(smoke_results) == 5
assert all(set(REQUIRED_METADATA).issubset(doc.metadata) for doc in smoke_results)
display(pd.DataFrame([{'rank': rank, 'similarity': round(doc.metadata['similarity'], 4), 'source': doc.metadata['source'], 'article_id': doc.metadata['article_id'], 'name': doc.metadata['name'], 'url': doc.metadata['url']} for rank, doc in enumerate(smoke_results, start=1)]))

,rank,similarity,source,article_id,name,url
0,1,0.6384,inven_tip,47118,[1.2.413] 전 레벨구간 스타포스 확정복구 사용 가이드 (샤타반영),https://www.inven.co.kr/board/maple/2304/47118
1,2,0.5803,inven_tip,43522,[스타포스] 신 강화 목표별 파괴확률 정리,https://www.inven.co.kr/board/maple/2304/43522
2,3,0.5545,inven_tip,43466,썬데이 / 패치후 메할일&변경사항요약+@ : NEXT 마지막 업데이트,https://www.inven.co.kr/board/maple/2304/43466
3,4,0.5509,guide,412,[장비] 스타포스 강화,https://maplestory.nexon.com/Guide/N23GameInfo...
4,5,0.5231,inven_tip,43410,[3/20 적용]개편&바뀌는 것들 요약 + 캐시 / 신규 이벤트,https://www.inven.co.kr/board/maple/2304/43410


In [6]:
report = {'created_at': datetime.now(timezone.utc).isoformat(), 'persist_directory': str(CHROMA_DIR), 'legacy_collection': LEGACY_COLLECTION_NAME, 'legacy_collection_preserved': LEGACY_COLLECTION_NAME in {item.name for item in client.list_collections()}, 'collection_name': COLLECTION_NAME, 'collection_count': collection.count(), 'distance_metric': 'cosine', 'stale_ids_deleted': len(stale_ids), 'source_counts': source_counts, 'required_metadata': list(REQUIRED_METADATA), 'metadata_validation_passed': True, 'embedding_model': MODEL_NAME, 'embedding_dimension': int(inven_manifest['embedding_dimension']), 'official_vector_source': official_vector_source, 'inven_vector_source': str(inven_embeddings_path), 'smoke_query': SMOKE_QUERY, 'smoke_top5': [{'source': doc.metadata['source'], 'article_id': doc.metadata['article_id'], 'name': doc.metadata['name'], 'url': doc.metadata['url'], 'similarity': round(doc.metadata['similarity'], 4)} for doc in smoke_results]}
atomic_write_json(REPORT_PATH, report)
print('통합 리포트:', REPORT_PATH)
print('팀 RAG Chain 연결: retrieve(question, k=5)')

통합 리포트: C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\data\RAG\maplestory_integrated_chromadb_report.json
팀 RAG Chain 연결: retrieve(question, k=5)
